[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/05-explainability/02-debugging_a_bad_match.ipynb)

# Debugging a Bad Match

Sooner or later, a query that looks like it should obviously work will return nothing, or will return the wrong candidate, or will silently drop a result you expected to see. This notebook walks through four real, reproducible causes, using the scoring behavior you learned in the previous notebook to actually diagnose each one, rather than guessing.

In this notebook you will:

1. See a perfect match on one field disappear entirely because of an unrelated field in the same query
2. Learn the diagnostic technique of isolating fields, one at a time, to find which one is causing a failure
3. Recognize when a short field has simply run out of fuzzy tolerance
4. Catch a candidate being silently filtered by `minimum_quality`, rather than genuinely not matching
5. Walk away with a practical checklist for your own debugging



In [1]:
# !pip install mbox

## 1. A perfect match on one field, gone entirely

Here is a product catalog, and a query where `product_name` is an exact, letter-for-letter match, but `description` is searched with something completely unrelated.

In [7]:
import pandas as pd
from mbox.indexing import TableIndexer

df = pd.DataFrame({
    "product_id": ["B88-EXT", "A12-PWR", "C99-SNS"],
    "product_name": ["Extended Battery Pack", "Portable Power Bank", "Motion Sensor Camera"],
    "description": [
        "Rechargeable power cell for outdoor gear",
        "Compact portable power bank for phones and tablets",
        "Motion sensor camera with night vision for home security"
    ]
})

df

,product_id,product_name,description
0,B88-EXT,Extended Battery Pack,Rechargeable power cell for outdoor gear
1,A12-PWR,Portable Power Bank,Compact portable power bank for phones and tab...
2,C99-SNS,Motion Sensor Camera,Motion sensor camera with night vision for hom...


Let's build an index and match

In [8]:
index = TableIndexer.create_index(df, index_columns=["product_name", "description"], tmp_dir="tmp_index")

result = index.match(
    product_name="Extended Battery Pack",
    description="Quantum flux capacitor housing",
    include_field_scores=True,
    min_total_match_value=0
)

result

,query_row,index_row,product_name_candidate,description_candidate,product_id_candidate,overall_score,product_name_score,description_score
0,0,-1,,,,0,0,0


`index_row` is `-1`. Every score column reads `0`, including `product_name_score`, even though `"Extended Battery Pack"` is a letter-for-letter match against the indexed data. This is not a low-scoring match, it is no match at all.

A multi-field query needs enough relevant signal in *every* field you search, not just one of them. `description="Quantum flux capacitor housing"` shares essentially nothing with any indexed description, and that single field's failure took the entire candidate down with it, regardless of how perfect `product_name` was.

## 2. Diagnosing which field is the problem

When a multi-field query returns nothing, the fastest way to find out which field is responsible is to query each field on its own, one at a time, and see which one still finds a candidate.

In [9]:
name_only = index.match(product_name="Extended Battery Pack", include_field_scores=True, min_total_match_value=0)
print("product_name alone:")
display(name_only)

description_only = index.match(description="Quantum flux capacitor housing", include_field_scores=True, min_total_match_value=0)
print("\ndescription alone:")
display(description_only)

product_name alone:


,query_row,index_row,product_name_candidate,description_candidate,product_id_candidate,overall_score,product_name_score
0,0,0,Extended Battery Pack,Rechargeable power cell for outdoor gear,B88-EXT,100,100



description alone:


,query_row,index_row,description_candidate,product_name_candidate,product_id_candidate,overall_score,description_score
0,0,-1,,,,0,0


`product_name` alone finds a perfect match, `product_name_score = 100`. `description` alone finds nothing at all. That immediately tells you where the problem is: not in your `product_name` query, not in your index, specifically in the `description` value you searched with.

This is the general technique worth remembering: **when a combined query fails, split it apart.** Query each field independently, and whichever one comes back empty on its own is the one to investigate first.

## 3. A short field that ran out of room

Recall from the previous notebook that shorter fields tolerate far fewer edits before the match disappears entirely. Here is that exact situation, on a real product ID.

In [4]:
id_index = TableIndexer.create_index(pd.DataFrame({"product_id": ["B88-EXT"]}), index_columns=["product_id"], tmp_dir="tmp_index")

test_queries = ["B88-EXT", "888-EXY", "888-EXZ", "888-EYZ"]

for q in test_queries:
    r = id_index.match(product_id=q, include_field_scores=True, min_total_match_value=0)
    found = len(r) > 0 and r["index_row"].iloc[0] != -1
    score = r["product_id_score"].iloc[0] if found else None
    print(f"{q!r:12s} found={found}   score={score}")

'B88-EXT'    found=True   score=100
'888-EXY'    found=True   score=36
'888-EXZ'    found=True   score=36
'888-EYZ'    found=False   score=None


`"B88-EXT"` is only 7 characters long. Two substitutions still find a match, with a low score. A third substitution, `"888-EYZ"`, loses the match completely.

If a colleague reports "this product ID search used to work and now it doesn't," and the ID field is short, this is one of the first things worth checking: not a bug, but a genuine, expected consequence of how little room a short field has for `APPROX` to absorb noise. This is exactly why `04-recall-tuning/01-understanding_match_modes.ipynb` recommends `IDENT` or `EXACT` for short identifiers, there is very little upside to `APPROX`'s fuzziness when the cutoff is this close.

## 4. A candidate silently filtered by `minimum_quality`

This one is the trickiest to catch, because there is no `-1` and no obviously broken score, the candidate simply is not in your results, and the reason is a rule you set yourself, possibly a while ago, in a different notebook or script.

In [10]:
from mbox.recall import TableRecallConfig, TableRecallFieldConfig, TableRecallMode

catalog = pd.DataFrame({
    "product_name": ["Extended Battery Pack", "Extended Cell Battery"],
    "description": [
        "Compact accessory offering extended battery life for outdoor gear",
        "Extended battery pack accessory for compact cameras"
    ]
})
catalog_index = TableIndexer.create_index(catalog, index_columns=["product_name", "description"], tmp_dir="tmp_index")

gated_config = TableRecallConfig(
    fields=[
        TableRecallFieldConfig(input_column="product_name", indexed_column="product_name",
                                minimum_quality=70, weight=50, mode=TableRecallMode.APPROX),
        TableRecallFieldConfig(input_column="description", indexed_column="description",
                                minimum_quality=0, weight=50, mode=TableRecallMode.APPROX)
    ],
    max_results=5,
    min_total_match_value=0,
    include_field_scores=True
)

gated_results = catalog_index.match(
    queries=pd.DataFrame({"product_name": ["Extended Battery Pack"], "description": ["Extended Battery Pack"]}),
    config=gated_config
)

print("With minimum_quality=70 on product_name:")
display(gated_results)

With minimum_quality=70 on product_name:


,query_row,index_row,product_name_candidate,description_candidate,overall_score,product_name_score,description_score
0,0,0,Extended Battery Pack,Compact accessory offering extended battery li...,64,100,29


Only one candidate comes back. Let's remove the `minimum_quality` gate and run the identical query, to see what was actually being filtered out.

In [11]:
ungated_config = TableRecallConfig(
    fields=[
        TableRecallFieldConfig(input_column="product_name", indexed_column="product_name",
                                minimum_quality=0, weight=50, mode=TableRecallMode.APPROX),
        TableRecallFieldConfig(input_column="description", indexed_column="description",
                                minimum_quality=0, weight=50, mode=TableRecallMode.APPROX)
    ],
    max_results=5,
    min_total_match_value=0,
    include_field_scores=True
)

ungated_results = catalog_index.match(
    queries=pd.DataFrame({"product_name": ["Extended Battery Pack"], "description": ["Extended Battery Pack"]}),
    config=ungated_config
)

print("Without the gate:")
display(ungated_results)

Without the gate:


,query_row,index_row,product_name_candidate,description_candidate,overall_score,product_name_score,description_score
0,0,0,Extended Battery Pack,Compact accessory offering extended battery li...,64,100,29
1,0,1,Extended Cell Battery,Extended battery pack accessory for compact ca...,64,42,86


`"Extended Cell Battery"` was there all along, with a perfectly respectable `overall_score = 64`. The `minimum_quality=70` floor on `product_name` excluded it, because its `product_name_score` of `42` did not clear that bar, even though nothing else about the match was wrong.

This is why `04-recall-tuning/02-weighting_fields_for_better_precision.ipynb` calls `minimum_quality` a hard floor rather than a discount. If you inherit a `TableRecallConfig` from somewhere else, a shared file, an older notebook, a teammate's code, and a candidate you expect to see is missing, checking every field's `minimum_quality` setting should be high on your list, right alongside checking `index_row`.

## 5. A practical debugging checklist

When a query does not return what you expect, work through these in order:

1. **Check `index_row`.** If it is `-1`, no candidate cleared the threshold for at least one searched field. This is not a scoring problem, it is a candidacy problem.
2. **Isolate each field.** Query every field in your multi-field search on its own. Whichever one comes back empty independently is where the problem lives.
3. **Check field length.** If the problematic field is short, three or four characters, consider whether `APPROX` simply ran out of room, rather than assuming something is broken.
4. **Check every `minimum_quality` setting.** If you are using a `TableRecallConfig`, especially one you did not write yourself moments ago, re-run the same query with every `minimum_quality` set to `0` and compare. A candidate that reappears was being filtered, not failing to match.
5. **Check `min_total_match_value`.** This is the same idea as `minimum_quality`, but applied to `overall_score` as a whole rather than one field. A perfectly reasonable candidate can still be excluded if its `overall_score` falls under this global floor.

Every one of these is something you can check directly, by running one more query, rather than guessing. That is the entire point of explainability: a bad match is not a black box, it is something you can take apart.

## Next steps

You now have the full explainability toolkit: how scores are actually calculated, and how to diagnose a match that does not behave the way you expected. From here:

- **`06-agentic-ai/`** - use M|BOX as a grounded, explainable tool inside LLM and agent workflows, where being able to justify a match is not just convenient, it is often the entire point

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*